# 第 8 讲实验：线性模型、LDA 与 Logistic Regression
先读工作单。学生只填写 `student-response` 单元。测试默认关闭。

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, confusion_matrix
from sklearn.base import clone
DATA=Path('..')/'data'/'heart.csv'
if not DATA.exists(): DATA=Path('class2026/materials/8. Linear-models/data/heart.csv')
df=pd.read_csv(DATA)
target='heart_disease_present'
features=['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal']
X=df[features].copy(); y=df[target].astype(int)
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=2026)
print(df.shape, y.value_counts().to_dict(), X_dev.shape, X_test.shape)

## A 数据审计
记录变量类型、结局比例和为什么 `record_id/site/num` 不进入模型。

In [ ]:
num_cols=['age','trestbps','chol','thalach','oldpeak']
cat_cols=[c for c in features if c not in num_cols]
pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_cols),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat_cols)])
models={'logistic':LogisticRegression(max_iter=3000),'lda':LinearDiscriminantAnalysis(),'gaussian_nb':GaussianNB()}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=2026)
def metric_row(name, p, yy, t=.5):
    pred=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(yy,pred,labels=[0,1]).ravel()
    return {'model':name,'roc_auc':roc_auc_score(yy,p),'pr_auc':average_precision_score(yy,p),'brier':brier_score_loss(yy,p),'log_loss':log_loss(yy,p),'sensitivity':tp/(tp+fn) if tp+fn else np.nan,'specificity':tn/(tn+fp) if tn+fp else np.nan,'FN':fn,'FP':fp,'cost_5FN_FP':5*fn+fp}
rows=[]
for name,est in models.items():
    pipe=Pipeline([('pre',pre),('model',est)])
    p=cross_val_predict(pipe,X_dev,y_dev,cv=cv,method='predict_proba')[:,1]
    rows.append(metric_row(name,p,y_dev))
prior=y_dev.mean(); rows.append(metric_row('prevalence_baseline',np.repeat(prior,len(y_dev)),y_dev))
results=pd.DataFrame(rows).sort_values('cost_5FN_FP'); results

## B 阈值与错误代价
学生先预想：阈值下降会怎样？再用 development 预测选择 `5×FN+FP` 最小的阈值。

In [ ]:
chosen_name='logistic'  # 先依据假设和目标写出理由，再修改此处。
if chosen_name=='prevalence_baseline': p_dev=np.repeat(prior,len(y_dev))
else:
    pipe=Pipeline([('pre',pre),('model',models[chosen_name])]); p_dev=cross_val_predict(pipe,X_dev,y_dev,cv=cv,method='predict_proba')[:,1]
threshold_rows=[metric_row(chosen_name,p_dev,y_dev,t) | {'threshold':t} for t in [.1,.2,.3,.4,.5,.6,.7,.8,.9]]
thresholds=pd.DataFrame(threshold_rows).sort_values(['cost_5FN_FP','threshold']); thresholds

## C 系数与稳定性
只对 logistic 拟合一个开发模型。解释一个 OR，并通过 bootstrap 观察系数变化。不要把 OR 写成因果效应。

In [ ]:
log_pipe=Pipeline([('pre',pre),('model',LogisticRegression(max_iter=3000))]).fit(X_dev,y_dev)
names=log_pipe.named_steps['pre'].get_feature_names_out(); coef=log_pipe.named_steps['model'].coef_[0]
coef_table=pd.DataFrame({'feature':names,'beta':coef,'OR':np.exp(coef)}).sort_values('OR',ascending=False); coef_table.head(10)

## D 冻结后打开测试
将 `OPEN_LOCKED_TEST` 改为 True 前，先提交模型、预处理、阈值、错误代价和指标。打开后不得根据结果修改方案。

In [ ]:
OPEN_LOCKED_TEST=False
if OPEN_LOCKED_TEST:
    final_model=clone(log_pipe).fit(X_dev,y_dev)
    p_test=final_model.predict_proba(X_test)[:,1]
    chosen=float(thresholds.iloc[0].threshold)
    print(metric_row('locked_test',p_test,y_test,chosen))
else:
    print('Test locked: freeze the protocol before opening.')